# Notebook 5 — Full End-to-End Flow

Everything wired together. Tests 10 real queries through the complete pipeline.

**Exit criteria:** 9/10 queries correct. SQL queries < 500ms. RAG < 5 seconds.

## 1. Setup — Load All Components

In [ ]:

import sqlite3
import numpy as np
import re
import time
import json
from datetime import datetime

DB_PATH = "second_brain.db"
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# Load embedding model
try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    EMB_AVAILABLE = True
    print("Embedding model loaded.")
except ImportError:
    EMB_AVAILABLE = False
    print("sentence-transformers not available — RAG tests will be skipped")

TODAY = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
CURRENT_MONTH = TODAY[:7]
print(f"Today: {TODAY}")
print(f"Current month: {CURRENT_MONTH}")


## 2. Paste Parser from Notebook 2

> Copy the `parse_amount`, `tag_category`, `parse_single`, `parse_note`, and `get_confirmation_message` functions here, or import from a .py file.

In [ ]:
# ── Inline parser (mirror of Notebook 2 — keep in sync if NB2 changes) ──
# No category. Person whitelist gates weight detection. Loaded fresh from `persons` table.

def load_persons_from_db():
    cur.execute("SELECT name FROM persons")
    return {r['name'] for r in cur.fetchall()}

KNOWN_PERSONS = load_persons_from_db()
print("KNOWN_PERSONS loaded from DB:", sorted(KNOWN_PERSONS))

GAVE_KEYWORDS     = r'\b(gave|give|given|lent|sent|advanced)\b'
RECEIVED_KEYWORDS = r'\b(got|received|returned|paid back)\b'
PERSON_CMD_RE = re.compile(
    r'^(ADD_PERSON|REMOVE_PERSON|MODIFY_PERSON)\s*:\s*(.+)$',
    re.IGNORECASE,
)

def parse_amount(text):
    text = text.strip().replace(',', '')
    m = re.match(r'^(\d+\.?\d*)\s*[kK]$', text)
    if m: return float(m.group(1)) * 1000
    m = re.match(r'^(\d+\.?\d*)\s*[lL]$', text)
    if m: return float(m.group(1)) * 100000
    m = re.match(r'^(\d+\.?\d*)$', text)
    if m: return float(m.group(1))
    return None

def find_amount_in_tokens(tokens):
    for i, t in enumerate(tokens):
        a = parse_amount(t)
        if a is not None: return a, i
    return None, -1

def get_month(date_str): return date_str[:7]

def parse_person_command(text):
    m = PERSON_CMD_RE.match(text.strip())
    if not m: return None
    op = m.group(1).upper()
    payload = m.group(2).strip().lower()
    if op == 'MODIFY_PERSON':
        parts = payload.split()
        if len(parts) != 2:
            return {'type': 'person_command', 'op': op,
                    'error': 'Expected: MODIFY_PERSON: oldname newname', 'raw': text}
        return {'type': 'person_command', 'op': op,
                'old_name': parts[0], 'new_name': parts[1], 'raw': text}
    return {'type': 'person_command', 'op': op, 'name': payload, 'raw': text}

def parse_single(text, today):
    text = text.strip()
    if not text: return None
    tl = text.lower()
    tokens = text.split()

    cmd = parse_person_command(text)
    if cmd: return cmd

    if re.search(r'\bgift\b', tl):
        amt, _ = find_amount_in_tokens(tokens)
        if amt:
            desc = re.sub(r'\b\d+[kKlL]?\b|gift', '', text).strip()
            return {'type': 'expense', 'amount': amt, 'description': desc or 'gift',
                    'date': today, 'month': get_month(today), 'raw': text}

    if re.search(RECEIVED_KEYWORDS, tl):
        pm = re.search(r'from\s+([a-zA-Z]+)', tl) or re.search(r'^([a-zA-Z]+)\s+(?:returned|paid)', tl)
        amt, _ = find_amount_in_tokens(tokens)
        if amt and pm:
            person = pm.group(1).lower()
            return {'type': 'ledger', 'person': person, 'amount': amt,
                    'direction': 'received', 'date': today, 'raw': text,
                    'unknown_person': person not in KNOWN_PERSONS}

    if re.search(GAVE_KEYWORDS, tl):
        gm = re.search(r'(?:gave|give|given|lent|sent|advanced)\s+([a-zA-Z]+)', tl)
        pgm = re.search(r'^([a-zA-Z]+)\s+(?:gave|give)', tl)
        amt, _ = find_amount_in_tokens(tokens)
        if amt:
            if pgm:
                person = pgm.group(1).lower()
                return {'type': 'ledger', 'person': person, 'amount': amt,
                        'direction': 'received', 'date': today, 'raw': text,
                        'unknown_person': person not in KNOWN_PERSONS}
            elif gm:
                person = gm.group(1).lower()
                return {'type': 'ledger', 'person': person, 'amount': amt,
                        'direction': 'gave', 'date': today, 'raw': text,
                        'unknown_person': person not in KNOWN_PERSONS}

    # Weight: STRICT — only persons in KNOWN_PERSONS
    for name in KNOWN_PERSONS:
        m = re.search(rf'\b{name}\b\s+(\d+\.?\d*)|(\d+\.?\d*)\s+\b{name}\b', tl)
        if m:
            raw_num = m.group(1) or m.group(2)
            w = float(raw_num)
            if w < 150:
                nm = re.search(rf'\b{name}\b\s+\d+\.?\d*\s*(.*)', tl)
                note = nm.group(1).strip() if nm and nm.group(1).strip() else None
                return {'type': 'weight', 'person': name, 'weight': w,
                        'note': note, 'date': today, 'raw': text}

    amt, amt_idx = find_amount_in_tokens(tokens)
    if amt is not None:
        desc = ' '.join(t for i, t in enumerate(tokens) if i != amt_idx).strip(' -+') or 'misc'
        return {'type': 'expense', 'amount': amt, 'description': desc,
                'date': today, 'month': get_month(today), 'raw': text}

    return {'type': 'todo', 'content': text, 'date': None, 'raw': text}

def parse_note(raw_input, today=None):
    if today is None: today = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    raw_input = raw_input.strip()
    cmd = parse_person_command(raw_input)
    if cmd: return [cmd]
    parts = [p.strip() for p in raw_input.split(',') if p.strip()]
    return [r for p in parts for r in [parse_single(p, today)] if r]

print("Parser loaded.")

## 3. Tool Implementations

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_notes_tool(query, domain, top_k=3):
    if not EMB_AVAILABLE:
        return "Embedding model not available"
    query_emb = emb_model.encode(query).astype(np.float32)
    cur.execute("SELECT content, embedding, source FROM embeddings WHERE domain = ?", (domain,))
    rows = cur.fetchall()
    if not rows: return "No notes found in this domain"
    results = [(cosine_similarity(query_emb, np.frombuffer(r['embedding'], dtype=np.float32)), r['content'], r['source']) for r in rows]
    results.sort(reverse=True)
    top = results[:top_k]
    return "\n---\n".join([f"[{src}] {content}" for _, content, src in top])

def query_ledger_tool(person=None, query_type="balance"):
    if query_type == "balance" and person:
        cur.execute("SELECT balance FROM ledger_balance WHERE person = ?", (person.lower(),))
        row = cur.fetchone()
        if row:
            bal = row['balance']
            if bal > 0: return f"{person.title()} owes you ₹{bal:,.0f}"
            elif bal < 0: return f"You owe {person.title()} ₹{abs(bal):,.0f}"
            else: return f"{person.title()} — all settled"
        return f"No ledger entries found for {person}"
    
    elif query_type == "who_owes":
        cur.execute("SELECT person, balance FROM ledger_balance WHERE balance > 0 ORDER BY balance DESC")
        rows = cur.fetchall()
        if not rows: return "Nobody owes you money"
        return "\n".join([f"{r['person'].title()}: ₹{r['balance']:,.0f}" for r in rows])
    
    return "Unknown query type"

def query_expense_tool(month=None, description_like=None):
    where, params = [], []
    if month:
        where.append("month = ?"); params.append(month)
    if description_like:
        where.append("description LIKE ?"); params.append(f"%{description_like}%")
    sql = "SELECT SUM(amount) as total FROM expenses"
    if where: sql += " WHERE " + " AND ".join(where)
    cur.execute(sql, params)
    row = cur.fetchone()
    total = row['total'] if row and row['total'] else 0
    parts = []
    if month: parts.append(f"month {month}")
    if description_like: parts.append(f"description ~ '{description_like}'")
    label = " + ".join(parts) if parts else "all time"
    return f"Total spend ({label}): ₹{total:,.0f}"

def get_todos_tool(status="pending"):
    cur.execute("SELECT content FROM todos WHERE status=?", (status,))
    rows = cur.fetchall()
    if not rows: return f"No {status} todos"
    return "\n".join([f"• {r['content']}" for r in rows])

def get_weight_tool(person, limit=1):
    cur.execute("SELECT weight, date, note FROM weights WHERE person=? ORDER BY date DESC LIMIT ?", (person.lower(), limit))
    rows = cur.fetchall()
    if not rows: return f"No weight data for {person}"
    if limit == 1:
        r = rows[0]
        note = f" ({r['note']})" if r['note'] else ""
        return f"{person.title()} weight: {r['weight']}kg on {r['date'][:10]}{note}"
    return "\n".join([f"{r['date'][:10]}: {r['weight']}kg" + (f" ({r['note']})" if r['note'] else "") for r in rows])

def manage_persons_tool(cmd):
    """
    cmd is a parsed person_command dict from the parser.
    Mutates the `persons` table AND refreshes the in-memory KNOWN_PERSONS set.
    """
    global KNOWN_PERSONS
    if 'error' in cmd:
        return f"⚠ {cmd['error']}"
    op = cmd['op']
    try:
        if op == 'ADD_PERSON':
            name = cmd['name']
            cur.execute("INSERT INTO persons (name) VALUES (?)", (name,))
            conn.commit()
            KNOWN_PERSONS = load_persons_from_db()
            return f"✓ Added '{name}' to people"
        if op == 'REMOVE_PERSON':
            name = cmd['name']
            cur.execute("DELETE FROM persons WHERE name = ?", (name,))
            conn.commit()
            removed = cur.rowcount
            KNOWN_PERSONS = load_persons_from_db()
            return f"✓ Removed '{name}'" if removed else f"⚠ '{name}' was not in people list"
        if op == 'MODIFY_PERSON':
            old, new = cmd['old_name'], cmd['new_name']
            cur.execute("UPDATE persons SET name = ? WHERE name = ?", (new, old))
            conn.commit()
            updated = cur.rowcount
            KNOWN_PERSONS = load_persons_from_db()
            return f"✓ Renamed '{old}' → '{new}'" if updated else f"⚠ '{old}' was not in people list"
    except sqlite3.IntegrityError as e:
        return f"⚠ {e}"
    return "Unknown person command"

def add_entry_tool(parsed_entry):
    t = parsed_entry['type']
    if t == 'expense':
        cur.execute(
            "INSERT INTO expenses (amount, description, date, month, raw_note) VALUES (?,?,?,?,?)",
            (parsed_entry['amount'], parsed_entry['description'],
             parsed_entry['date'], parsed_entry['month'], parsed_entry['raw'])
        )
        conn.commit()
        return f"✓ Added expense: ₹{parsed_entry['amount']:.0f} — {parsed_entry['description']}"
    elif t == 'ledger':
        cur.execute(
            "INSERT INTO ledger (person, amount, direction, date) VALUES (?,?,?,?)",
            (parsed_entry['person'], parsed_entry['amount'], parsed_entry['direction'], parsed_entry['date'])
        )
        conn.commit()
        verb = "Gave" if parsed_entry['direction'] == 'gave' else "Received from"
        msg = f"✓ {verb} {parsed_entry['person'].title()} ₹{parsed_entry['amount']:.0f} logged"
        if parsed_entry.get('unknown_person'):
            msg += f". Tip: ADD_PERSON: {parsed_entry['person']} to track future entries cleanly"
        return msg
    elif t == 'weight':
        cur.execute(
            "INSERT INTO weights (person, weight, date, note) VALUES (?,?,?,?)",
            (parsed_entry['person'], parsed_entry['weight'], parsed_entry['date'], parsed_entry.get('note'))
        )
        conn.commit()
        return f"✓ {parsed_entry['person'].title()} weight: {parsed_entry['weight']}kg logged"
    elif t == 'todo':
        cur.execute("INSERT INTO todos (content) VALUES (?)", (parsed_entry['content'],))
        conn.commit()
        return f"✓ Todo added: {parsed_entry['content']}"
    elif t == 'person_command':
        return manage_persons_tool(parsed_entry)
    return "Unknown entry type"

print("All tools defined (incl. manage_persons_tool).")

## 4. Query Router

In [ ]:
def route_query(query, today, current_month):
    """
    Route a natural language query to the right tool.
    Returns (tool_name, result, latency_ms)
    """
    q = query.lower().strip()
    start = time.time()
    
    # ── PERSON COMMAND (highest priority — mutate whitelist) ──
    cmd = parse_person_command(query)
    if cmd:
        result = manage_persons_tool(cmd)
        return 'manage_persons', result, (time.time()-start)*1000

    # ── WEIGHT QUERIES ────────────────────────────────────
    # Only people who actually have weight history (small subset of KNOWN_PERSONS)
    weight_persons = {'jeevi', 'prani', 'murugan'}
    for name in weight_persons:
        if name in q and any(w in q for w in ['weight', 'latest', 'last', 'trend', 'how much', 'kg']):
            limit = 5 if 'trend' in q or 'last 5' in q else 1
            return 'get_weight', get_weight_tool(name, limit), (time.time()-start)*1000
        if name in q and 'weight' not in q and 'balance' not in q:
            return 'get_weight', get_weight_tool(name, 1), (time.time()-start)*1000

    # ── LEDGER BALANCE ────────────────────────────────────
    if 'who owes' in q or 'owes me' in q:
        return 'query_ledger', query_ledger_tool(query_type="who_owes"), (time.time()-start)*1000
    
    for name in KNOWN_PERSONS:
        if name in q and any(w in q for w in ['balance', 'owe', 'owes', 'how much']):
            return 'query_ledger', query_ledger_tool(person=name, query_type="balance"), (time.time()-start)*1000

    # ── EXPENSE QUERIES (no category — use description LIKE) ─
    if any(w in q for w in ['spend', 'spent', 'expense', 'spending']):
        month = None
        description_like = None
        month_map = {'january':'2026-01','february':'2026-02','march':'2026-03',
                     'april':'2026-04','may':'2026-05','feb':'2026-02','mar':'2026-03',
                     'jan':'2026-01','apr':'2026-04'}
        for word, m in month_map.items():
            if word in q: month = m; break
        if not month and 'this month' in q:
            month = current_month
        for kw in ['petrol', 'diesel', 'medicine', 'medplus', 'pampers', 'electricity',
                   'broadband', 'biryani', 'tea', 'rice', 'mutton', 'food', 'groceries']:
            if kw in q: description_like = kw; break
        return 'query_expense', query_expense_tool(month=month, description_like=description_like), (time.time()-start)*1000

    # ── TODOS ─────────────────────────────────────────────
    if any(w in q for w in ['todo', 'task', 'pending', 'tasks']):
        return 'get_todos', get_todos_tool('pending'), (time.time()-start)*1000

    # ── INVESTMENT RAG ────────────────────────────────────
    investment_signals = ['anand', 'stock', 'invest', 'pharma', 'cipla', 'natco', 'bank',
                          'mistake', '48 hour', 'peter lynch', 'allocation', 'portfolio', 'pe ratio']
    if any(s in q for s in investment_signals):
        return 'search_notes', search_notes_tool(query, 'investment', top_k=2), (time.time()-start)*1000

    # ── HEALTH RAG ────────────────────────────────────────
    health_signals = ['eat', 'drink', 'tomato', 'brinjal', 'milk', 'snack',
                      'psoriasis', 'histamine', 'gut', 'avoid', 'safe']
    if any(s in q for s in health_signals):
        return 'search_notes', search_notes_tool(query, 'health', top_k=2), (time.time()-start)*1000

    # ── ADD ENTRY (note input) ────────────────────────────
    parsed = parse_note(query, today)
    if parsed and parsed[0]['type'] != 'todo':
        results = [add_entry_tool(p) for p in parsed]
        return 'add_entry', '\n'.join(results), (time.time()-start)*1000

    return 'unknown', f"Could not route query: {query}", (time.time()-start)*1000

print("Router defined.")

## 5. Run 10 Real Test Queries

In [ ]:
test_queries = [
    # (query, expected_tool, expected_keyword_in_result)
    ("Maddy balance",                          "query_ledger",  "7,000"),
    ("Who owes me money",                      "query_ledger",  "Maddy"),
    ("Jeevi latest weight",                    "get_weight",    "60.1"),
    ("How much did I spend in February",       "query_expense", "3,506"),
    ("How much did I spend on petrol",         "query_expense", "petrol"),
    ("Pending todos",                          "get_todos",     "electricity"),
    ("gave Mani 2000",                         "add_entry",     "Mani"),       # unknown person → nudge
    ("petrol 500",                             "add_entry",     "petrol"),
    ("ADD_PERSON: ravi",                       "manage_persons", "Added"),     # whitelist mutation
    ("What did Anand say about Cipla",         "search_notes",  "Cipla"),
    ("Can I eat tomato",                       "search_notes",  "nightshade"),
]

print(f"{'Query':<45} {'Tool':<17} {'Correct':>8} {'Latency':>10}")
print("="*85)

passed = 0
total = len(test_queries)

for query, expected_tool, expected_keyword in test_queries:
    tool, result, latency = route_query(query, TODAY, CURRENT_MONTH)
    
    tool_ok = tool == expected_tool
    result_ok = expected_keyword.lower() in result.lower()
    both_ok = tool_ok and result_ok
    
    if both_ok:
        passed += 1
    
    status = "✓" if both_ok else "✗"
    latency_str = f"{latency:.0f}ms"
    print(f"{status} {query:<43} {tool:<17} {'Yes' if both_ok else 'No':>8} {latency_str:>10}")
    
    if not both_ok:
        if not tool_ok:
            print(f"  Tool: expected={expected_tool}, got={tool}")
        if not result_ok:
            print(f"  Result missing '{expected_keyword}'")
            print(f"  Got: {result[:100]}")

print(f"\n{'='*85}")
print(f"Passed: {passed}/{total}")
print(f"EXIT CRITERIA: >= {int(total*0.9)}/{total}")
print("PASS ✓" if passed >= int(total*0.9) else "FAIL ✗ — fix routing or tool logic")

## 6. Latency Summary

In [ ]:

print("Latency benchmark:")
print("="*50)

latency_tests = [
    ("SQL query (Maddy balance)",      "Maddy balance"),
    ("SQL query (Feb spend)",          "How much did I spend in February"),
    ("SQL query (pending todos)",      "Pending todos"),
    ("Write (add expense)",            "petrol 500"),
    ("RAG (investment)",               "What did Anand say about Cipla"),
    ("RAG (health)",                   "Can I eat tomato"),
]

for label, query in latency_tests:
    times = []
    for _ in range(3):  # run 3 times, take average
        _, _, ms = route_query(query, TODAY, CURRENT_MONTH)
        times.append(ms)
    avg = sum(times) / len(times)
    threshold = 5000 if 'RAG' in label else 500
    status = "✓" if avg < threshold else "✗ SLOW"
    print(f"{status} {label:<35} avg={avg:.0f}ms (threshold: {threshold}ms)")

conn.close()
print("\nAll notebooks complete. Ready for Android port.")
